### Data :

       +) /kaggle/input/datasets/adityajn105/flickr30k

       +) /kaggle/input/datasets/adityajn105/flickr8k

       +)/kaggle/input/datasets/leo040802/ktvic-dataset

       +) /kaggle/input/datasets/awsaf49/coco-2017-dataset



**IMPORT LIBARY**

In [ ]:
import os
import re
import json
import math
import random
import gc
import subprocess
from copy import deepcopy
from itertools import cycle
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init
from torch.utils.data import Dataset, DataLoader, BatchSampler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from timm import create_model
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from pycocotools.coco import COCO
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from IPython.display import FileLink, display
from sklearn.model_selection import train_test_split


try:
    from pycocoevalcap.eval import COCOEvalCap
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.cider.cider import Cider
except ImportError:
    print("[INFO] Installing pycocoevalcap...")
    !pip install pycocoevalcap
    from pycocoevalcap.eval import COCOEvalCap
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.cider.cider import Cider


try:
    from helper_functions import set_seeds
except ImportError:
    print("[INFO] Downloading helper_functions...")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/helper_functions.py .
    !rm -rf pytorch-deep-learning
    from helper_functions import set_seeds

**TOKENIZER**

In [ ]:
set_seeds()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Id ngôn ngữ
lang_id = {
    "vi": 0, 
    "en": 1
}

# Load Tokenizers
tokenizers = {
    lang_id["vi"]: AutoTokenizer.from_pretrained("vinai/phobert-base"),
    lang_id["en"]: AutoTokenizer.from_pretrained("gpt2")
}

# Cập nhật special tokens cho GPT-2
special_tokens = {
    "bos_token": "<s>", 
    "eos_token": "</s>", 
    "pad_token": "<pad>"
}
tokenizers[1].add_special_tokens(special_tokens)

# Tạo dictionary lưu thông tin vocab & token đặc biệt
vocab_sizes =   { lid: len(tokenizer) for lid, tokenizer in tokenizers.items() }
pad_token_ids = { lid: tokenizer.pad_token_id for lid, tokenizer in tokenizers.items() }
bos_token_ids = { lid: tokenizer.bos_token_id for lid, tokenizer in tokenizers.items() }
eos_token_ids = { lid: tokenizer.eos_token_id for lid, tokenizer in tokenizers.items() }

**LOAD DATA & PREPROCESS CAPTION**

In [ ]:
def preprocess_caption(text):
    # Chuẩn hóa chuỗi về chữ thường
    text = text.lower()
    # Xóa ký tự đặc biệt, giữ lại chữ và số
    text = re.sub(r'[^\w\s]', '', text)
    # Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_flickr_data(input_directory):
    # Load data Flickr 8k/30k
    captions_path = os.path.join(input_directory, "captions.txt")
    try:
        image_path = os.path.join(input_directory, "Images")
    except:
        image_path = os.path.join(input_directory, "Images/flickr30k_images")

    df = pd.read_csv(captions_path, sep=',', header=None, names=['image', 'caption'], on_bad_lines='skip')
    df = df[1:].dropna().reset_index(drop=True) 

    df["image"] = df["image"].apply(lambda x: os.path.join(image_path, x))
    df["caption"] = df["caption"].apply(lambda x: f"<s> {preprocess_caption(x)}</s>")
    df["language"] = "en"

    return df

def load_coco_data(input_directory, split="train"):
    # Load data COCO 2017 train/val
    if split == "train":
        images_dir = os.path.join(input_directory, "train2017")
        annotations_path = os.path.join(input_directory, "annotations/captions_train2017.json")
    elif split == "val":
        images_dir = os.path.join(input_directory, "val2017")
        annotations_path = os.path.join(input_directory, "annotations/captions_val2017.json")
    else:
        raise ValueError("Not 'train'/'val'")

    coco = COCO(annotations_path)
    data = [
        {
            "image": f"{images_dir}/{img_info['file_name']}",
            "caption": f"<s> {preprocess_caption(ann['caption'])}</s>",
            "language": "en"
        }
        for image_id in coco.getImgIds()
        for img_info in coco.loadImgs(image_id)
        for ann in coco.loadAnns(coco.getAnnIds(imgIds=image_id))
    ]
    return pd.DataFrame(data)

def load_ktvic_data(input_directory):
    # Load data tiếng Việt KTVIC
    train_path = os.path.join(input_directory, "ktvic_dataset/train_data.json")
    test_path = os.path.join(input_directory, "ktvic_dataset/test_data.json")

    with open(train_path, "r", encoding="utf-8") as f:
        train_json = json.load(f)
    
    with open(test_path, "r", encoding="utf-8") as f:
        test_json = json.load(f)

    # Map id ảnh với tên file
    train_images = {img["id"]: img["filename"] for img in train_json["images"]}
    test_images = {img["id"]: img["filename"] for img in test_json["images"]}

    # Load data train
    train_data = []
    for ann in train_json["annotations"]:
        image_id = ann["image_id"]
        if image_id in train_images:
            image_path = os.path.join(input_directory, "ktvic_dataset/train-images", train_images[image_id])
            train_data.append({
                "image": image_path, 
                "caption": preprocess_caption(ann["caption"]),
                "language": "vi"
            })

    # Load data test
    test_data = []
    for ann in test_json["annotations"]:
        image_id = ann["image_id"]
        if image_id in test_images:
            image_path = os.path.join(input_directory, "ktvic_dataset/public-test-images", test_images[image_id])
            test_data.append({
                "image": image_path, 
                "caption": preprocess_caption(ann["caption"]),
                "language": "vi"
            })

    df_train = pd.DataFrame(train_data)
    df_test = pd.DataFrame(test_data)

    # Clean ảnh không tồn tại
    df_train = df_train[df_train["image"].apply(os.path.exists)].reset_index(drop=True)
    df_test = df_test[df_test["image"].apply(os.path.exists)].reset_index(drop=True)

    return df_train, df_test

def split_by_caption_length(df):
    # Phân loại độ khó dựa trên số lượng từ trong caption
    df_easy = df[df["caption"].apply(lambda x: len(x.split()) < 10)]
    df_medium = df[df["caption"].apply(lambda x: 10 <= len(x.split()) < 20)]
    df_hard = df[df["caption"].apply(lambda x: len(x.split()) >= 20)]
    return df_easy, df_medium, df_hard
    
def load_data(dataset_type):
    if dataset_type == "coco-2017":
        return load_coco_data("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017", split="train"), load_coco_data("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017", split="val")
    if dataset_type == "flickr30k":
        return load_flickr_data("/kaggle/input/datasets/adityajn105/flickr30k")
    if dataset_type == "flickr8k":
        return load_flickr_data("/kaggle/input/datasets/adityajn105/flickr8k")
    if dataset_type == "ktvic":
        return load_ktvic_data("/kaggle/input/datasets/leo040802/ktvic-dataset")
    return None

In [ ]:
def captions_length(data, title):
    plt.figure(figsize=(15, 7), dpi=300)
    sns.set_style('darkgrid') 

    lengths = [len(x.split(' ')) for x in data] 

    sns.histplot(x=lengths, kde=True, binwidth=1)

    plt.title(f'Captions Length Histogram ({title})', fontsize=15, fontweight='bold')
    plt.xlabel('Length (Number of Words)', fontweight='bold')
    plt.ylabel('Frequency', fontweight='bold')

    plt.xticks(fontweight='bold')
    plt.yticks(fontweight='bold')

    plt.show()

**DATASET & DATA LOADER**

In [ ]:
class ImageCaptionDataset(Dataset):
    def __init__(self, dataframe, image_transform, tokenizers, lang_id_map, max_length=30, multi_caption=False):
        self.data = dataframe
        self.image_transform = image_transform
        self.tokenizers = tokenizers
        self.lang_id_map = lang_id_map
        self.max_length = max_length
        self.multi_caption = multi_caption # Val

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Read data từ dataframe
        img_path, captions, language = self.data.iloc[idx][["image", "caption", "language"]]
        lang_id = self.lang_id_map[language]
        tokenizer = self.tokenizers[lang_id]

        # Đọc ảnh và convert RGB
        image = Image.open(img_path).convert("RGB")
        image = self.image_transform(image)

        # Xử lý Train
        if not self.multi_caption:
            tokens = tokenizer(
                captions,
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
                add_special_tokens=True
            )
            return image, tokens["input_ids"].squeeze(0), torch.tensor(lang_id)

        # Xử lý Val
        token_ids_list = []
        for caption in captions: 
            tokens = tokenizer(
                caption,
                truncation=True,
                padding="max_length",
                max_length=self.max_length,
                return_tensors="pt",
                add_special_tokens=True
            )
            token_ids_list.append(tokens["input_ids"].squeeze(0))
            
        return image, token_ids_list, torch.tensor(lang_id)

# Cấu hình Transform ảnh
max_length = 30
image_size = 192
batch_size = 128

image_transform = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize ImageNet
])

# Gom all caption 1 ảnh vào một list
def group_captions(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby(["image", "language"])["caption"].apply(list).reset_index()

# Load data
train_vi, raw_test_vi = load_data("ktvic") 
train_coco, val_coco = load_data("coco-2017")
train_fl = load_data("flickr30k")

# Gộp data
train_en = pd.concat([train_coco, train_fl]).reset_index(drop=True)
val_vi, test_vi = train_test_split(raw_test_vi, test_size=0.5, random_state=42)
val_en, test_en = train_test_split(val_coco, test_size=0.5, random_state=42)

df_train = {
    "vi": train_vi,
    "en": train_en
}

df_val = {
    "vi": group_captions(val_vi),
    "en": group_captions(val_en)
}

df_test = {
    "vi": group_captions(test_vi),
    "en": group_captions(test_en)
}

dataset_splits = {}
train_loaders = {}
val_loaders = {}
test_loaders = {}

# Chia DataLoader Train theo độ dài caption (easy, medium, hard)
for lang, df in df_train.items():
    df_easy, df_medium, df_hard = split_by_caption_length(df)
    
    dataset_splits[lang] = {
        "easy": ImageCaptionDataset(df_easy, image_transform, tokenizers, lang_id, max_length, multi_caption=False),
        "medium": ImageCaptionDataset(df_medium, image_transform, tokenizers, lang_id, max_length, multi_caption=False),
        "hard": ImageCaptionDataset(df_hard, image_transform, tokenizers, lang_id, max_length, multi_caption=False)
    }

    train_loaders[lang] = {
        level: DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=os.cpu_count(),
            pin_memory=True,
            prefetch_factor=8,
            drop_last=True
        ) for level, dataset in dataset_splits[lang].items()
    }

def val_collate_fn(batch):
    images, token_ids, lang_ids = zip(*batch)
    return torch.stack(images), token_ids, torch.stack(lang_ids)

# DataLoader Val
for lang, df in df_val.items():
    dataset = ImageCaptionDataset(df, image_transform, tokenizers, lang_id, max_length, multi_caption=True)
    val_loaders[lang] = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=os.cpu_count(),
        pin_memory=True,
        prefetch_factor=8,
        drop_last=True,
        collate_fn=val_collate_fn
    )

# DataLoader Test
for lang, df in df_test.items():
    dataset = ImageCaptionDataset(df, image_transform, tokenizers, lang_id, max_length, multi_caption=True)
    test_loaders[lang] = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=os.cpu_count(),
        pin_memory=True,
        prefetch_factor=8,
        drop_last=True,
        collate_fn=val_collate_fn
    )

In [ ]:
def get_coco_captions_by_filename(input_directory, filename, split="train"):

    if split == "train":
        annotations_path = os.path.join(input_directory, "annotations/captions_train2017.json")
    elif split == "val":
        annotations_path = os.path.join(input_directory, "annotations/captions_val2017.json")
    else:
        raise ValueError("Not 'train'/'val'")

    # Load dữ liệu annotations COCO
    coco = COCO(annotations_path)

    # Tìm image_id trên filename
    image_id = None
    for img in coco.dataset["images"]:
        if img["file_name"] == filename:
            image_id = img["id"]
            break

    # Lấy danh sách captions dựa vào image_id
    ann_ids = coco.getAnnIds(imgIds=image_id)
    annotations = coco.loadAnns(ann_ids)

    # Lấy tối đa 5 captions
    captions = [ann["caption"] for ann in annotations[:5]]

    return captions
input_directory = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
filename = "000000002685.jpg"
captions = get_coco_captions_by_filename(input_directory, filename, split="val")

print("Captions:", captions)


**SWINV2 & TRANSFORMER DECODER**

In [ ]:
# Token Embedding
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim, pad_token_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_token_id)
        self.scale = math.sqrt(embedding_dim)

    def forward(self, tokens):
        return self.embedding(tokens) * self.scale

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, dropout=0.1, maxlen=1024):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        den = torch.exp(-torch.arange(0, embedding_dim, 2) * math.log(10000.0) / embedding_dim)
        pos = torch.arange(0, maxlen).unsqueeze(1)
        pos_embedding = torch.zeros(maxlen, embedding_dim)
        pos_embedding[:, 0::2] = torch.sin(pos * den)
        pos_embedding[:, 1::2] = torch.cos(pos * den)
        self.register_buffer('pos_embedding', pos_embedding.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pos_embedding[:, :x.size(1), :])

class DAEProjection(nn.Module):
    def __init__(self, input_dim=1536, latent_dim=1024):
        super().__init__()

        # Feature Weighting
        self.feature_weight = nn.Parameter(torch.ones(1, input_dim))

        # LayerNorm
        self.norm = nn.LayerNorm(input_dim)

        # Encoder : Giảm chiều
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1408),  
            nn.GELU(),
            nn.Linear(1408, 1280),  
            nn.GELU(),
            nn.Linear(1280, latent_dim)
        )

        # Self-Attention
        self.attn = nn.MultiheadAttention(embed_dim=latent_dim, num_heads=16, batch_first=True)

        # Residual Connection
        self.residual_weight = nn.Parameter(torch.tensor(0.1))  

        # Decoder : Khôi phục
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1280),
            nn.GELU(),
            nn.Linear(1280, 1408),
            nn.GELU(),
            nn.Linear(1408, input_dim)
        )

    def forward(self, x):
        x = self.norm(x) * self.feature_weight

        # Encoder
        z = self.encoder(x)
        z, _ = self.attn(z, z, z)

        # Decoder
        reconstructed = self.decoder(z) + x * self.residual_weight

        # MSE Loss
        recon_loss = F.mse_loss(reconstructed, x, reduction='mean')

        return z, reconstructed, recon_loss

# Image Caption Model 
class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_sizes, pad_token_ids, embedding_dim=1024, num_heads=16, num_decoder_layers=8, ffn_dim=4096, dropout=0.1):
        super().__init__()
        self.pad_token_ids = pad_token_ids

        # Encoder : Swinv2
        self.swin_encoder = create_model("swinv2_large_window12_192", pretrained=True, num_classes=0)

        # Self-Attention
        self.mhsa = nn.MultiheadAttention(embed_dim=1536, num_heads=num_heads, dropout=0.1, batch_first=True)

        # DAE Projection
        self.projection = DAEProjection(1536, embedding_dim)

        # Positional Encoding
        self.positional_encodings = nn.ModuleList([
            PositionalEncoding(embedding_dim, dropout=0.1) for _ in vocab_sizes
        ])


        # Token Embeddings
        self.token_embeddings = nn.ModuleList([
            TokenEmbedding(vocab_sizes[lang], embedding_dim, pad_token_ids[lang]) for lang in vocab_sizes
        ])

        # List Decoders
        self.decoders = nn.ModuleList([
            nn.TransformerDecoder(
                nn.TransformerDecoderLayer(d_model=embedding_dim, nhead=num_heads, dim_feedforward=ffn_dim, dropout=dropout, norm_first=True, batch_first=True, activation="gelu"),
                num_layers=num_decoder_layers
            ) for _ in vocab_sizes
        ])

        # Generator List
        self.generators = nn.ModuleList([
            nn.Linear(embedding_dim, vocab_sizes[lang]) for lang in vocab_sizes
        ])

    def encode(self, img):
        with torch.no_grad():
            memory = self.swin_encoder.forward_features(img)

        b, h, w, c = memory.shape
        memory = memory.view(b, h * w, c)

        # Self-Attention
        memory, _ = self.mhsa(memory, memory, memory)

        # Giảm chiều Deterministic AE
        reduced_memory, reconstructed, dae_loss = self.projection(memory)

        return reduced_memory, dae_loss

    def forward(self, img, tokens, lang_id):
        device = tokens.device
        lang_idx = lang_id[0].item()

        memory, dae_loss = self.encode(img)

        tgt_emb = self.token_embeddings[lang_idx](tokens)
        tgt_emb = self.positional_encodings[lang_idx](tgt_emb)

        seq_len = tokens.shape[1]
        mask = torch.triu(torch.ones((seq_len, seq_len), device=device) == 1).transpose(0, 1)
        tgt_mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        padding_mask = (tokens == self.pad_token_ids[lang_idx]).float().to(device)

        decoded = self.decoders[lang_idx](tgt_emb, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=padding_mask)

        return self.generators[lang_idx](decoded), dae_loss.unsqueeze(0)


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# Đường dẫn lưu checkpoint
input_path = "/kaggle/input/multilingual_model/pytorch/default/1/multilingual_model.pth"
checkpoint_path = "/kaggle/working/multilingual_model_new.pth"

def init_weights(module):    
    if isinstance(module, nn.Linear):
        if isinstance(module, nn.GELU):
            init.kaiming_uniform_(module.weight, nonlinearity='relu')
        else:
            init.xavier_uniform_(module.weight)
        if module.bias is not None:
            init.zeros_(module.bias)

    elif isinstance(module, nn.Embedding):
        init.trunc_normal_(module.weight, mean=0, std=0.02)

    elif isinstance(module, nn.LayerNorm):
        module.bias.data.zero_()
        module.weight.data.fill_(1.0)

    elif isinstance(module, nn.MultiheadAttention):
        if hasattr(module, "in_proj_weight") and module.in_proj_weight is not None:
            init.xavier_uniform_(module.in_proj_weight)
        if hasattr(module, "q_proj_weight") and module.q_proj_weight is not None:
            init.xavier_uniform_(module.q_proj_weight)
        if hasattr(module, "k_proj_weight") and module.k_proj_weight is not None:
            init.xavier_uniform_(module.k_proj_weight)
        if hasattr(module, "v_proj_weight") and module.v_proj_weight is not None:
            init.xavier_uniform_(module.v_proj_weight)
        if hasattr(module, "out_proj") and module.out_proj.weight is not None:
            init.xavier_uniform_(module.out_proj.weight)

    elif isinstance(module, nn.Parameter):
        init.uniform_(module, a=0.0, b=1.0)

def apply_init_weights(model):
    if isinstance(model, torch.nn.DataParallel):
        model = model.module

    model.mhsa.apply(init_weights)
    model.token_embeddings.apply(init_weights)

    model.positional_encodings.apply(init_weights)

    model.projection.apply(init_weights)
    model.generators.apply(init_weights)

def save_checkpoint(model, optimizers_state, schedulers_state, scalers_state, start_epochs, train_losses):
    checkpoint = {
        "start_epochs": start_epochs,
        "model_state_dict": model.module.state_dict(),
        "optimizers_state": optimizers_state,
        "schedulers_state": schedulers_state,
        "scalers_state": scalers_state,
        "train_losses": train_losses,
        "vocab_sizes": vocab_sizes,
        "pad_token_ids": pad_token_ids,
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

def create_optimizer_for_lang(model, lang, device):
    lang_idx = lang_id[lang]
    shared_params = (
        list(model.module.swin_encoder.parameters()) +
        list(model.module.mhsa.parameters()) +
        list(model.module.projection.parameters())
    )
    lang_specific_params = (
        list(model.module.positional_encodings[lang_idx].parameters()) + 
        list(model.module.token_embeddings[lang_idx].parameters()) +
        list(model.module.decoders[lang_idx].parameters()) +
        list(model.module.generators[lang_idx].parameters())
    )
    # optimizer = torch.optim.AdamW(shared_params + lang_specific_params, lr=1e-4)
    # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=1)
    optimizer = torch.optim.AdamW(
        shared_params + lang_specific_params,
        lr=3e-4,
        weight_decay=0.01,
        betas=(0.9, 0.98),
        eps=1e-8
    )

    # CosineAnnealing + warmup
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=1, eta_min=1e-6)
    scaler = torch.amp.GradScaler()
    return optimizer, scheduler, scaler

def load_optimizer_for_lang(checkpoint, model, lang, device):
    lang_idx = lang_id[lang]
    if checkpoint and lang in checkpoint["optimizers_state"]:
        optimizer, scheduler, scaler = create_optimizer_for_lang(model, lang, device)
        optimizer.load_state_dict(checkpoint["optimizers_state"][lang])
        scheduler.load_state_dict(checkpoint["schedulers_state"][lang])
        scaler.load_state_dict(checkpoint["scalers_state"][lang])
        return optimizer, scheduler, scaler
    else:
        return create_optimizer_for_lang(model, lang, device)

def load_checkpoint(device):
    if os.path.exists(input_path):

        checkpoint = torch.load(input_path, map_location="cpu", weights_only=False)
        model = ImageCaptionModel(
            vocab_sizes=checkpoint["vocab_sizes"],
            pad_token_ids=checkpoint["pad_token_ids"]
        )
        apply_init_weights(model)

        model.load_state_dict(checkpoint["model_state_dict"], strict=False)

        state_dict = checkpoint["model_state_dict"]
        old_pe_key = "positional_encoding.pos_embedding"
        if any(k.startswith(old_pe_key) for k in state_dict):
            old_pe_value = state_dict[old_pe_key]
            for i, pe in enumerate(model.positional_encodings):
                with torch.no_grad():
                    pe.pos_embedding.copy_(old_pe_value.clone())

        model.to(device)

        if torch.cuda.device_count() > 1:
            model = torch.nn.DataParallel(model)

        print("Model loaded successfully.")

        start_epochs = checkpoint.get("start_epochs", {lang: 1 for lang in lang_id.keys()})
        train_losses = checkpoint.get("train_losses", {lang: [] for lang in lang_id.keys()})

        return model, checkpoint, start_epochs, train_losses

    else:
        print("No checkpoint found.")
        model = ImageCaptionModel(vocab_sizes, pad_token_ids).to(device)
        if torch.cuda.device_count() > 1:
            model = torch.nn.DataParallel(model)

        apply_init_weights(model)

        start_epochs = {lang: 1 for lang in lang_id.keys()}
        train_losses = {lang: [] for lang in lang_id.keys()}

        checkpoint = {
            "start_epochs": start_epochs,
            "train_losses": train_losses,
            "optimizers_state": {},
            "schedulers_state": {},
            "scalers_state": {},
            "vocab_sizes": vocab_sizes,
            "pad_token_ids": pad_token_ids,
        }

        return model, checkpoint, start_epochs, train_losses
model, checkpoint, start_epochs, train_losses = load_checkpoint(device)

**TRAIN MODEL**

In [ ]:
def train_model_per_language(
    model,
    train_loader,
    lang,
    lang_id,
    optimizer,
    scaler,
    start_epochs,
    accumulation_steps=1,
):
    lang_idx = lang_id[lang]
    criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_token_ids[lang_idx], label_smoothing=0.01)

    model.train()
    total_loss, step = 0.0, 0
    loop = tqdm(train_loader, desc=f"Epoch {start_epochs} ({lang})", unit="batch")
    
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (img, tokens, lang_id_batch) in enumerate(loop):
        img = img.to(device, non_blocking=True)
        tokens = tokens.to(device, non_blocking=True)
        lang_id_batch = lang_id_batch.to(device, non_blocking=True)

        decoder_input = tokens[:, :-1]
        decoder_target = tokens[:, 1:]

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            output, recon_loss = model(img, decoder_input, lang_id_batch)
            lm_loss = criterion(output.view(-1, output.size(-1)), decoder_target.reshape(-1))
            recon_loss = recon_loss.mean()
            loss = lm_loss + recon_loss

        loss = loss / accumulation_steps
        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            
            torch.nn.utils.clip_grad_norm_(
                list(model.module.swin_encoder.parameters()) +  
                list(model.module.mhsa.parameters()) +  
                list(model.module.projection.parameters()) + 
                list(model.module.positional_encodings[lang_idx].parameters()) +
                list(model.module.token_embeddings[lang_idx].parameters()) +  
                list(model.module.decoders[lang_idx].parameters()) +  
                list(model.module.generators[lang_idx].parameters()), 
                max_norm=5.0
            )
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accumulation_steps
        step += 1
        loop.set_postfix(avg_loss=total_loss / step, loss=lm_loss.item(), recon=recon_loss.item())

    return total_loss / step


# Cấu hình và Huấn luyện
lang = 'en'
num_rounds = 25

optimizer, scheduler, scaler = load_optimizer_for_lang(checkpoint, model, lang, device)

# Train
for round_idx in range(num_rounds):
    print(f"\nRound {round_idx + 1}/{num_rounds}")

    avg_loss = 0.0
    levels = ["easy", "medium", "hard"]
    
    for level in levels:
        print(f"Training {level} captions...")
        
        avg_loss += train_model_per_language(
            model=model,
            train_loader=train_loaders[lang][level],
            lang=lang,
            lang_id=lang_id,
            optimizer=optimizer,
            scaler=scaler,
            start_epochs=start_epochs[lang],
            accumulation_steps=1 
        )

    avg_loss /= len(levels)
    start_epochs[lang] += 1  
    train_losses[lang].append(avg_loss)

    try:
        scheduler.step(avg_loss) 
    except TypeError:
        scheduler.step()
    
    # Lưu trạng thái
    checkpoint["optimizers_state"][lang] = optimizer.state_dict()
    checkpoint["schedulers_state"][lang] = scheduler.state_dict()
    checkpoint["scalers_state"][lang] = scaler.state_dict()
    
    # Lưu checkpoint
    save_checkpoint(
        model,
        checkpoint["optimizers_state"],
        checkpoint["schedulers_state"],
        checkpoint["scalers_state"],
        start_epochs,
        train_losses
    )

**EVALUATION**

In [ ]:
@torch.no_grad()
def predict_caption_batch_greedy(model, images, tokenizer, lang_ids, max_length=30):
    """
    Hàm sinh caption cho batch ảnh.
    """
    model.eval()
    device = images.device
    batch_size = images.size(0)

    bos_token_id = tokenizer.bos_token_id
    eos_token_id = tokenizer.eos_token_id

    # Tạo ma trận token <s> [batch_size, 1]
    generated_tokens = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)
    
    # Đánh dấu câu trong batch đã gặp token </s>
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for _ in range(max_length):
        output, _ = model(images, generated_tokens, lang_ids)
        
        # Chỉ lấy xác suất của token ở bước thời gian cuối cùng
        next_token_logits = output[:, -1, :]
        
        # Lấy ID của token có xác suất cao nhất
        next_token_ids = next_token_logits.argmax(dim=-1).unsqueeze(-1)

        # Nối token mới dự đoán vào chuỗi hiện tại
        generated_tokens = torch.cat([generated_tokens, next_token_ids], dim=-1)
        
        # Nếu sinh </s> đánh dấu câu True
        finished |= (next_token_ids.squeeze(-1) == eos_token_id)

        # Nếu tất cả các câu trong batch đều đã có </s> : Thoát vòng lặp
        if finished.all():
            break

    # Trả về list văn bản đã được decode
    return [tokenizer.decode(tokens.tolist(), skip_special_tokens=True) for tokens in generated_tokens]

@torch.no_grad()
def export_coco_eval_json(model, test_loader, tokenizer, output_dir="./eval_json", lang="vi"):
    """
    Hàm xuất kết quả dự đoán (res) và nhãn gốc (gts)
    """
    os.makedirs(output_dir, exist_ok=True)
    res, annotations, images = [], [], []
    device = next(model.parameters()).device

    # Lọc ra tập hợp các ID token đặc biệt
    special_tokens = {
        tok for tok in [
            tokenizer.pad_token_id, tokenizer.unk_token_id,
            tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.mask_token_id
        ] if tok is not None
    }

    cap_id, img_id = 0, 0

    for batch_idx, (images_tensor, token_ids_list, lang_ids) in enumerate(tqdm(test_loader, desc=f"Evaluating {lang.upper()}")):
        images_tensor = images_tensor.to(device)
        lang_ids = lang_ids.to(device)
        
        # Predict caption
        predictions = predict_caption_batch_greedy(model, images_tensor, tokenizer, lang_ids)

        # Save data
        for i, (pred_caption, ref_token_set) in enumerate(zip(predictions, token_ids_list)):
            # Save câu dự đoán
            res.append({"image_id": img_id, "caption": pred_caption.strip()})

            if isinstance(ref_token_set, torch.Tensor):
                ref_token_set = [ref_token_set]

            # Save list các reference (nhãn gốc) của cùng 1 ảnh (gts)
            for ref_tokens in ref_token_set:
                ref_tokens = ref_tokens.to(device)
                
                # Bỏ các token <pad>, <s>, </s>
                valid = ref_tokens[~torch.isin(ref_tokens, torch.tensor(list(special_tokens), device=device))]
                decoded = tokenizer.decode(valid, skip_special_tokens=True)
                
                if decoded:
                    annotations.append({"image_id": img_id, "id": cap_id, "caption": decoded})
                    cap_id += 1

            images.append({"id": img_id})
            img_id += 1

    # Tạo data cho Ground Truth (gts)
    gts = {"images": images, "annotations": annotations, "type": "captions"}

    res_path = f"{output_dir}/res_{lang}.json"
    gts_path = f"{output_dir}/gts_{lang}.json"

    with open(res_path, "w", encoding="utf-8") as f: json.dump(res, f, indent=2, ensure_ascii=False)
    with open(gts_path, "w", encoding="utf-8") as f: json.dump(gts, f, indent=2, ensure_ascii=False)

    print(f"Đã xuất file: {res_path}, {gts_path}")
    return gts_path, res_path

def coco_eval(gts_json_path, res_json_path):
    """
    Tính chỉ số BLEU, METEOR, ROUGE_L, CIDEr (pycocoevalcap)
    """
    coco = COCO(gts_json_path)
    cocoRes = coco.loadRes(res_json_path)
    cocoEval = COCOEvalCap(coco, cocoRes)
    
    cocoEval.evaluate()
    
    print("\nFINAL EVALUATION")
    for metric, score in cocoEval.eval.items():
        print(f"{metric}: {score:.4f}")
    return cocoEval.eval

def predict_caption(model, image, tokenizer, lang_id, max_length=30, beam_size=3):
    """
    Dự đoán caption cho 1 ảnh (Beam Search).
    """
    model.eval()
    device = next(model.parameters()).device 

    # Thêm batch dimension [1, C, H, W]
    image = image.unsqueeze(0).to(device)

    bos_token_id = tokenizer.bos_token_id
    eos_token_id = tokenizer.eos_token_id

    # Danh sách chứa tuple (chuỗi_token, điểm_số_tích_lũy)
    beams = [(torch.tensor([[bos_token_id]], dtype=torch.long, device=device), 0.0)]
    completed_beams = [] # Chứa các câu đã gặp token </s>

    for step in range(max_length):
        new_beams = []
        for seq, score in beams:
            # Nếu câu đã kết thúc ở vòng trước, append vào completed và bỏ qua
            if seq[0, -1].item() == eos_token_id:
                completed_beams.append((seq, score))
                continue

            with torch.no_grad():
                output, _ = model(image, seq, torch.tensor([lang_id], dtype=torch.long, device=device))
            
            # Lấy xác suất token kế tiếp và chuyển log_softmax
            logits = output[:, -1, :] 
            probs = F.log_softmax(logits, dim=-1) 
            top_k = torch.topk(probs, beam_size, dim=-1) 

            # Nối từng token trong top K vào chuỗi hiện tại
            for i in range(beam_size):
                next_token_id = top_k.indices[0, i].item()
                next_score = score + top_k.values[0, i].item() # Cộng điểm log
                
                new_seq = torch.cat([seq, torch.tensor([[next_token_id]], device=device)], dim=-1)
                new_beams.append((new_seq, next_score))

        # Sort các nhánh mới sinh theo điểm giảm dần, giữ lại beam_size nhánh có điểm score tích lũy cao nhất
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]
        
        # Nếu tất cả các nhánh có xác xuất tích lũy cao nhất đều đã sinh token id(</s>) -> Break
        if all(seq[0, -1].item() == eos_token_id for seq, _ in beams):
            break

    # Gom các câu đã hoàn tất
    completed_beams.extend(beams)
    
    # Chọn ra chuỗi có score tích lũy cao nhất
    best_seq = sorted(completed_beams, key=lambda x: x[1], reverse=True)[0][0]

    return tokenizer.decode(best_seq.squeeze().tolist(), skip_special_tokens=True)

lang_en = 'en'
gts_en, res_en = export_coco_eval_json(model, test_loaders[lang_en], tokenizers[lang_id[lang_en]], lang=lang_en)
coco_eval(gts_en, res_en)

lang_vi = 'vi'
gts_vi, res_vi = export_coco_eval_json(model, test_loaders[lang_vi], tokenizers[lang_id[lang_vi]], lang=lang_vi)
coco_eval(gts_vi, res_vi)

# Test 1 image
image_path = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/test2017/000000000001.jpg"
image = Image.open(image_path).convert("RGB")
image = image_transform(image) 

caption_vi = predict_caption(model, image, tokenizers[0], lang_id=0, beam_size=3)
print("Tiếng Việt (Beam=3):", caption_vi)

caption_en = predict_caption(model, image, tokenizers[1], lang_id=1, beam_size=3)
print("Tiếng Anh (Beam=3):", caption_en)

In [ ]:
def download_file(path, download_file_name):
    os.chdir('/kaggle/working/')
    zip_name = f"/kaggle/working/{download_file_name}.zip"
    command = f"zip {zip_name} {path} -r"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("Unable to run zip command!")
        print(result.stderr)
        return
    display(FileLink(f'{download_file_name}.zip'))
    
download_file('/kaggle/working/multilingual_model_old.pth', 'multilingual_model_old')